<a href="https://colab.research.google.com/github/Saniya9873/Coding-Performance-Analytics-Leaderboard-System/blob/main/Data_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
from google.colab import files
uploaded = files.upload()

Saving bigdata_learning_analytics.csv to bigdata_learning_analytics.csv
Saving Leetcode.csv to Leetcode.csv


In [3]:
import pandas as pd

lc = pd.read_csv("Leetcode.csv")
learn = pd.read_csv("bigdata_learning_analytics.csv")

In [4]:
print(lc.columns)
print(learn.columns)

Index(['ID', 'Title', 'Difficulty', 'Link', 'Topics', 'Acceptance Rate (%)',
       'Premium Only', 'Category', 'Likes', 'Dislikes', 'Example Test Cases',
       'Similar Questions'],
      dtype='object')
Index(['Student_ID', 'login_frequency', 'avg_session_duration',
       'completed_assignments', 'quiz_scores_avg', 'forum_participation',
       'resource_views', 'task_completion_time', 'task_accuracy',
       'error_count', 'repetition_needed', 'motion_intensity',
       'safety_violations', 'attention_span_score', 'fatigue_level',
       'collaboration_index', 'performance_score', 'efficiency_gain',
       'instruction_recommendation'],
      dtype='object')


In [5]:
print(lc.head())
print(lc.shape)

print(learn.head())

print(learn.shape)

     ID                                            Title Difficulty  \
0  3641                  Longest Semi-Repeating Subarray     Medium   
1  3642               Find Books with Polarized Opinions       Easy   
2  3633  Earliest Finish Time for Land and Water Rides I       Easy   
3  3647                       Maximum Weight in Two Bags     Medium   
4  3626             Find Stores with Inventory Imbalance     Medium   

                                                Link  \
0  https://leetcode.com/problems/longest-semi-rep...   
1  https://leetcode.com/problems/find-books-with-...   
2  https://leetcode.com/problems/earliest-finish-...   
3  https://leetcode.com/problems/maximum-weight-i...   
4  https://leetcode.com/problems/find-stores-with...   

                                              Topics  Acceptance Rate (%)  \
0                                                NaN                71.01   
1                                                NaN                50.77   
2  Ar

In [6]:
lc = lc[['Title', 'Difficulty', 'Topics']].copy()
lc.columns = ['problem', 'difficulty', 'topic']

In [7]:
learn = learn[['Student_ID', 'performance_score', 'quiz_scores_avg',
               'task_accuracy', 'error_count']].copy()

learn.columns = ['user_id', 'performance_score', 'quiz_score',
                 'task_accuracy', 'error_count']

In [10]:
# remove nulls
lc = lc.dropna(subset=['problem', 'difficulty'])

# topic clean
lc['topic'] = lc['topic'].astype(str).str.split(',').str[0].str.strip()

# difficulty standardize
lc['difficulty'] = lc['difficulty'].str.capitalize()

In [11]:
data = []
for _, row in learn.iterrows():
    user = row['user_id']
    perf = row['performance_score']
    quiz = row['quiz_score']
    accuracy = row['task_accuracy']
    errors = row['error_count']

    # submissions count based on performance
    num_submissions = int(np.random.randint(70, 130) + perf/2)

    # difficulty bias based on performance
    if perf >= 80:
        diff_probs = [0.3, 0.45, 0.25]   # more hard
        solve_prob = 0.85
    elif perf >= 60:
        diff_probs = [0.45, 0.4, 0.15]
        solve_prob = 0.75
    else:
        diff_probs = [0.6, 0.3, 0.1]
        solve_prob = 0.65

    for i in range(num_submissions):

        chosen_diff = np.random.choice(['Easy','Medium','Hard'], p=diff_probs)

        subset = lc[lc['difficulty'] == chosen_diff]
        if len(subset) == 0:
            subset = lc

        problem_row = subset.sample(1).iloc[0]

        status = np.random.choice(['Solved','Attempted'], p=[solve_prob, 1-solve_prob])

        # time based on difficulty + errors
        base_time = {
            'Easy': np.random.randint(5, 20),
            'Medium': np.random.randint(15, 40),
            'Hard': np.random.randint(30, 80)
        }[chosen_diff]

        time_taken = base_time + np.random.randint(0, errors + 5)

        data.append([
            user,
            pd.Timestamp('2025-01-01') + pd.Timedelta(days=np.random.randint(0,180)),
            problem_row['problem'],
            chosen_diff,
            problem_row['topic'],
            status,
            time_taken,
            perf,
            quiz,
            accuracy
        ])

In [14]:
df = pd.DataFrame(data, columns=[
    'user_id','date','problem','difficulty','topic','status',
    'time_taken','performance_score','quiz_score','task_accuracy'
])

df['date'] = pd.to_datetime(df['date'])
df = df[df['status']=='Solved']

In [15]:
difficulty_map = {'Easy':1, 'Medium':2, 'Hard':3}
df['difficulty_score'] = df['difficulty'].map(difficulty_map)

df['month'] = df['date'].dt.month

In [16]:
total = df.groupby('user_id').size().reset_index(name='total_solved')

consistency = df.groupby('user_id')['date'].nunique().reset_index(name='active_days')

difficulty_score = df.groupby('user_id')['difficulty_score'].sum().reset_index(name='difficulty_score')

topics = df.groupby('user_id')['topic'].nunique().reset_index(name='topics_covered')

monthly = df.groupby(['user_id','month']).size().reset_index(name='monthly_solved')

In [17]:
pivot = monthly.pivot(index='user_id', columns='month', values='monthly_solved').fillna(0)

if pivot.shape[1] >= 2:
    improvement = (pivot.iloc[:,-1] - pivot.iloc[:,-2]).reset_index(name='improvement')
else:
    improvement = pivot.sum(axis=1).reset_index(name='improvement')

In [18]:
metrics = total.merge(consistency,on='user_id') \
               .merge(difficulty_score,on='user_id') \
               .merge(topics,on='user_id') \
               .merge(improvement,on='user_id')

metrics = metrics.fillna(0)

In [19]:
def norm(col):
    return (col - col.min()) / (col.max() - col.min()) * 100

metrics['score'] = (
    0.25*norm(metrics['active_days']) +
    0.20*norm(metrics['difficulty_score']) +
    0.20*norm(metrics['topics_covered']) +
    0.20*norm(metrics['improvement']) +
    0.15*norm(metrics['total_solved'])
)

metrics = metrics.sort_values('score', ascending=False).reset_index(drop=True)
metrics['rank'] = metrics.index + 1

In [20]:
df.to_csv("final_dataset.csv", index=False)
metrics.to_csv("leaderboard.csv", index=False)

In [21]:
print(df.head())
print(metrics.head())
print(df.shape)
print(metrics.shape)

  user_id       date                                            problem  \
0    S001 2025-04-20           Most Visited Sector in  a Circular Track   
1    S001 2025-02-03             Find All K-Distant Indices in an Array   
4    S001 2025-04-14                          Patients With a Condition   
5    S001 2025-05-11  Count Partitions With Max-Min Difference at Mo...   
8    S001 2025-02-18  Partition Array Into Two Arrays to Minimize Su...   

  difficulty     topic  status  time_taken  performance_score  quiz_score  \
0       Easy     Array  Solved          12               65.4        75.7   
1       Easy     Array  Solved          20               65.4        75.7   
4       Easy  Database  Solved          23               65.4        75.7   
5     Medium     Array  Solved          16               65.4        75.7   
8       Hard     Array  Solved          86               65.4        75.7   

   task_accuracy  difficulty_score  month  
0           75.8                 1      4 

In [22]:
import os
print(os.listdir())

['.config', 'Leetcode.csv', 'leaderboard.csv', 'final_dataset.csv', 'bigdata_learning_analytics.csv', 'sample_data']


In [29]:
from google.colab import files
files.download("final_dataset.csv")
files.download("leaderboard.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [30]:
topic_perf = df.groupby(['user_id', 'topic']).size().reset_index(name='solved_count')
topic_perf.to_csv("topic_performance.csv", index=False)
files.download("topic_performance.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [31]:
difficulty_perf = df.groupby(['user_id', 'difficulty']).size().reset_index(name='solved_count')
difficulty_perf.to_csv("difficulty_performance.csv", index=False)
files.download("difficulty_performance.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [32]:
monthly_progress = df.groupby(['user_id', 'month']).size().reset_index(name='monthly_solved')
monthly_progress.to_csv("monthly_progress.csv", index=False)
files.download("monthly_progress.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [33]:
user_topic = df.groupby(['user_id', 'topic']).size().reset_index(name='solved_count')

avg_topic = user_topic.groupby('topic')['solved_count'].mean().reset_index(name='avg_solved')

recommendation = user_topic.merge(avg_topic, on='topic')
recommendation = recommendation[recommendation['solved_count'] < recommendation['avg_solved']].copy()
recommendation['recommendation'] = "Focus more on this topic"

recommendation.to_csv("recommendations.csv", index=False)
files.download("recommendations.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>